# Valorant

Neste notebook, vamos treinar um modelo YOLO usando um dataset do Roboflow Universe sobre Valorant.

Dataset usado no projeto: `https://universe.roboflow.com/fps/valorant-gygiq`


## 1. Verificando a Estrutura do Dataset Valorant

Vamos garantir que o `data.yaml` existe e que as pastas `train`, `valid` e `test` estão acessíveis.


In [ ]:
import os
import yaml
from pathlib import Path
from ultralytics import YOLO, settings

# Como este notebook fica dentro da pasta notebooks,
# usamos ../ para voltar para a raiz do projeto valorant-ai.
PROJECT_ROOT = Path("..").resolve()
DATASET_DIR = PROJECT_ROOT / "dataset_valorant"
dataset_yaml = DATASET_DIR / "data.yaml"

# Salvar os resultados na pasta runs da raiz do projeto
settings.update({"runs_dir": str(PROJECT_ROOT / "runs")})

print("Raiz do projeto:", PROJECT_ROOT)
print("Pasta do dataset:", DATASET_DIR)
print("Arquivo data.yaml:", dataset_yaml)

if dataset_yaml.exists():
    print("\n✅ Arquivo data.yaml encontrado!")
else:
    raise FileNotFoundError(f"❌ Arquivo NÃO encontrado: {dataset_yaml}")

# Conferindo as pastas principais do padrão YOLO
for split in ["train", "valid", "test"]:
    images_dir = DATASET_DIR / split / "images"
    labels_dir = DATASET_DIR / split / "labels"

    print(f"\n{split.upper()}:")
    print("images existe?", images_dir.exists(), "| qtd:", len(list(images_dir.glob("*"))) if images_dir.exists() else 0)
    print("labels existe?", labels_dir.exists(), "| qtd:", len(list(labels_dir.glob("*.txt"))) if labels_dir.exists() else 0)

# Lendo classes do data.yaml
with open(dataset_yaml, "r", encoding="utf-8") as f:
    data_config = yaml.safe_load(f)

print("\nClasses encontradas no data.yaml:")
print(data_config.get("names", "Nenhuma classe encontrada"))

print("\nConteúdo resumido do data.yaml:")
for key in ["path", "train", "val", "valid", "test", "nc", "names"]:
    if key in data_config:
        print(f"{key}: {data_config[key]}")


KeyboardInterrupt: 

## 2. Inicializando o Modelo e Treinando

Nós começamos com um modelo pré-treinado (`yolov8n.pt`) e ensinamos as classes do dataset de Valorant a ele passando o nosso `data.yaml`.

* `data`: caminho para o `data.yaml` do dataset Valorant.
* `epochs`: quantidade de épocas de treino. Para teste rápido, use 5.
* `imgsz`: tamanho da imagem. Usei 416 para ser mais leve no PC.
* `batch`: quantidade de imagens processadas por vez. Usei 2 para evitar erro de memória.

Se o treino estiver muito lento ou der erro de GPU, use `device="cpu"`.


In [ ]:
model = YOLO("yolov8n.pt")

# Treino no padrão da aula, usando o dataset de Valorant
results = model.train(
    data=str(dataset_yaml),
    epochs=30,
    imgsz=416,
    batch=2,
    name="valorant_yolo"
)

# Se der erro de memória/GPU, troque o treino acima por este:
# results = model.train(
#     data=str(dataset_yaml),
#     epochs=5,
#     imgsz=416,
#     batch=2,
#     name="valorant_yolo_cpu",
#     device="cpu"
# )


## 3. Analisando os Resultados 📊

Após o término do treino, os resultados serão salvos na pasta `runs/detect/valorant_yolo`.

Vamos mostrar:
* `val_batch0_pred.jpg`: imagem de validação com as caixas previstas.
* `results.png`: gráficos do treinamento.


In [ ]:
from IPython.display import Image, display
import glob

# Pegando a última pasta de treino criada para Valorant
list_of_runs = glob.glob(str(PROJECT_ROOT / "runs" / "detect" / "valorant_yolo*"))
latest_run = max(list_of_runs, key=os.path.getctime) if list_of_runs else None

if latest_run:
    print(f"Analisando resultados da pasta: {latest_run}")

    # Mostrar as predições em um lote de validação
    val_img = os.path.join(latest_run, "val_batch0_pred.jpg")
    if os.path.exists(val_img):
        print("\nAmostra de validação:")
        display(Image(filename=val_img, width=800))
    else:
        print("\nval_batch0_pred.jpg ainda não foi encontrado.")

    # Mostrar o gráfico de resultados
    results_img = os.path.join(latest_run, "results.png")
    if os.path.exists(results_img):
        print("\nGráficos de treinamento:")
        display(Image(filename=results_img, width=800))
    else:
        print("\nresults.png ainda não foi encontrado.")
else:
    print("Nenhum resultado de treino encontrado ainda.")


## 4. Testando o Modelo Treinado em Imagens do Teste

Depois do treino, usamos o `best.pt` gerado para fazer predições nas imagens da pasta `test/images`.


In [ ]:
best_model_path = Path(latest_run) / "weights" / "best.pt"

if best_model_path.exists():
    trained_model = YOLO(str(best_model_path))

    trained_model.predict(
        source=str(DATASET_DIR / "test" / "images"),
        conf=0.35,
        save=True,
        name="valorant_teste_imagens"
    )

    print("✅ Predições salvas na pasta runs/detect/valorant_teste_imagens")
else:
    print("❌ best.pt não encontrado. Execute o treino primeiro.")


## 5. Aplicação Prática: analisando um clipe de Valorant 🎬

Agora vamos usar o modelo treinado para analisar um vídeo real da pasta `clips`.

Nesta etapa, o notebook vai:

1. carregar o arquivo `best.pt`, que foi gerado no treinamento;
2. analisar o clipe escolhido frame por frame;
3. salvar um novo vídeo com as detecções desenhadas;
4. salvar os arquivos `.txt` com as detecções;
5. gerar prints e um resumo final para usar na entrega.

> **Observação:** como o dataset veio com classes numéricas, as caixas podem aparecer como `0` e `1`, em vez de nomes como Jett, Sage ou Raze.


### 5.1 Configuração do clipe e do modelo

Altere apenas o nome do clipe em `CLIP_NAME`, caso queira testar outro vídeo.

Exemplo:

```python
CLIP_NAME = "clip3.mp4"
```


In [ ]:
from pathlib import Path
from ultralytics import YOLO

# ==============================
# CONFIGURAÇÕES PRINCIPAIS
# ==============================
CLIP_NAME = "meu_clipe.mp4"          # Nome do vídeo dentro da pasta clips
CONFIDENCE = 0.10                # Confiança mínima da detecção
RUN_NAME = "clip_valorant_final" # Nome da pasta de saída em runs/detect

# Descobre a raiz do projeto automaticamente
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

CLIP_PATH = ROOT / "clips" / CLIP_NAME
RESULTS_DIR = ROOT / "runs" / "detect" / RUN_NAME

# Tenta usar a variável best_model_path criada nas etapas anteriores do notebook.
# Se ela não existir, usa o caminho padrão do treino principal.
try:
    MODEL_PATH = Path(best_model_path)
    if not MODEL_PATH.is_absolute():
        MODEL_PATH = ROOT / MODEL_PATH
except NameError:
    MODEL_PATH = ROOT / "runs" / "detect" / "valorant_yolo" / "weights" / "best.pt"

print("Projeto:", ROOT)
print("Clipe escolhido:", CLIP_PATH)
print("Modelo treinado:", MODEL_PATH)
print("Saída esperada:", RESULTS_DIR)

if not CLIP_PATH.exists():
    raise FileNotFoundError(f"Clipe não encontrado: {CLIP_PATH}")

if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Modelo treinado não encontrado: {MODEL_PATH}")


### 5.2 Rodando a análise do clipe

Esta célula executa o YOLO no vídeo escolhido.

Ela salva:

- o vídeo analisado em `runs/detect/clip_valorant_final`;
- as labels em `runs/detect/clip_valorant_final/labels`;
- as confianças das detecções junto das labels.


In [ ]:
model = YOLO(str(MODEL_PATH))

results = model.predict(
    source=str(CLIP_PATH),
    conf=CONFIDENCE,
    save=True,
    save_txt=True,
    save_conf=True,
    project=str(ROOT / "runs" / "detect"),
    name=RUN_NAME,
    exist_ok=True,
    verbose=False
)

print("Análise concluída!")
print("Resultado salvo em:", RESULTS_DIR)
print("Labels salvas em:", RESULTS_DIR / "labels")


### 5.3 Exibindo o vídeo analisado

O vídeo gerado já vem com as caixas de detecção desenhadas pelo YOLO.




In [ ]:
from IPython.display import Video, display

video_files = []
for ext in ["*.mp4", "*.avi", "*.mov", "*.mkv"]:
    video_files.extend(RESULTS_DIR.glob(ext))

print("Vídeos encontrados:")
for video in video_files:
    print("-", video)

if video_files:
    video_analisado = video_files[0]
    display(Video(str(video_analisado), embed=True, width=850))
else:
    raise FileNotFoundError(f"Nenhum vídeo analisado foi encontrado em: {RESULTS_DIR}")


### 5.4 Gerando prints do clipe analisado

Esta célula extrai alguns frames do vídeo analisado e mostra como prints no notebook.

Esses prints são úteis para colocar no relatório ou na apresentação da atividade.


In [ ]:
import cv2
from IPython.display import Image, display

prints_dir = ROOT / "runs" / "prints_clip_valorant"
prints_dir.mkdir(parents=True, exist_ok=True)

cap = cv2.VideoCapture(str(video_analisado))

if not cap.isOpened():
    raise RuntimeError(f"Não foi possível abrir o vídeo analisado: {video_analisado}")

fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
duracao = total_frames / fps if fps else 0

print(f"Vídeo analisado: {video_analisado.name}")
print(f"⏱Duração: {duracao:.2f} segundos")
print(f"Total de frames: {total_frames}")
print(f"FPS: {fps:.2f}")

# Quantidade de prints que serão mostrados
qtd_prints = 8

# Escolhe tempos igualmente espaçados ao longo do vídeo
if duracao > 0:
    tempos = [duracao * i / (qtd_prints + 1) for i in range(1, qtd_prints + 1)]
else:
    tempos = []

prints_salvos = []

for i, tempo in enumerate(tempos, start=1):
    frame_num = int(tempo * fps)
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_num)

    ret, frame = cap.read()
    if not ret:
        continue

    nome_print = prints_dir / f"print_{i:02d}_tempo_{tempo:.2f}s.jpg"
    cv2.imwrite(str(nome_print), frame)
    prints_salvos.append(nome_print)

cap.release()

print("Prints salvos em:", prints_dir)
print("Quantidade de prints:", len(prints_salvos))

for img in prints_salvos:
    print("\n", img.name)
    display(Image(filename=str(img), width=850))


### 5.5 Resumo das detecções do clipe

Aqui o notebook lê as labels salvas pelo YOLO e monta uma tabela com a quantidade de detecções por classe.

Como o dataset usado tem classes numéricas, o resumo aparece como classe `0`, classe `1`, etc.


In [ ]:
from collections import Counter, defaultdict
import pandas as pd

labels_dir = RESULTS_DIR / "labels"
label_files = sorted(labels_dir.glob("*.txt"))

contador = Counter()
confiancas = defaultdict(list)

for label_file in label_files:
    with open(label_file, "r", encoding="utf-8") as arquivo:
        for linha in arquivo:
            partes = linha.strip().split()

            if not partes:
                continue

            classe_id = partes[0]
            contador[classe_id] += 1

            # Quando save_conf=True, a confiança costuma vir na última coluna
            if len(partes) >= 6:
                try:
                    confiancas[classe_id].append(float(partes[-1]))
                except ValueError:
                    pass

linhas = []

for classe_id, quantidade in contador.most_common():
    nome_classe = model.names.get(int(classe_id), classe_id) if str(classe_id).isdigit() else classe_id
    media_conf = sum(confiancas[classe_id]) / len(confiancas[classe_id]) if confiancas[classe_id] else None

    linhas.append({
        "classe_id": classe_id,
        "nome_da_classe": nome_classe,
        "quantidade_de_deteccoes": quantidade,
        "confianca_media": round(media_conf, 4) if media_conf is not None else "-"
    })

if linhas:
    df_resumo = pd.DataFrame(linhas)
    display(df_resumo)
else:
    print("⚠️ Nenhuma detecção foi encontrada nos arquivos de labels.")
    print("Tente reduzir o CONFIDENCE ou testar outro clipe.")

print(f"📄 Arquivos de label encontrados: {len(label_files)}")


> Após o treinamento customizado do modelo YOLO com imagens do dataset de Valorant, foi realizado um teste prático em um clipe de gameplay. O modelo treinado foi carregado a partir do arquivo `best.pt` e aplicado frame a frame no vídeo. Como resultado, foi gerado um novo clipe com as caixas de detecção desenhadas, além de arquivos de label contendo as classes detectadas e suas respectivas coordenadas. Também foram extraídos prints do vídeo analisado para facilitar a visualização dos resultados obtidos.

> Como o dataset utilizado possui classes numéricas, as detecções aparecem identificadas como classe `0`, classe `1`, etc. Mesmo assim, o processo demonstra o funcionamento de um detector customizado treinado com YOLO para reconhecer elementos visuais específicos de Valorant.
